# Time Series Forecasting with Prophet
### Dataset: International Airline Passengers (1949-1960)

In this notebook, we will explore the fundamental concepts of Time Series Analysis and use **Facebook Prophet** to predict future passenger numbers. 

**Key Concepts covered:**
1. **Trend**: The long-term increase or decrease in the data.
2. **Seasonality**: Repeating patterns at fixed intervals (e.g., Summer peaks).
3. **Noise**: Random variations that cannot be explained by trend or seasonality.

In [9]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet

# Set display options
pd.options.display.max_rows = 10

## 1. Loading the Data
We are using a classic dataset representing the monthly total number of international airline passengers (in thousands).

In [2]:
url = "https://storage.googleapis.com/edulabs-public-datasets/airline-passengers.csv"
df = pd.read_csv(url)

# Check the first few rows
df.head()

,Month,Passengers
0,1949-01,112
1,1949-02,118
2,1949-03,132
3,1949-04,129
4,1949-05,121


## 2. Visualizing the Raw Data
Before modeling, we must always look at the data. Notice the clear upward trend and the repeating seasonal 'spikes'.

In [3]:
fig = px.line(df, x='Month', y='Passengers', 
              title='International Airline Passengers (1949-1960)',
              template='plotly_dark')
fig.update_traces(line_color='#00D4FF')
fig.show()

## 3. Preparing for Prophet
Prophet is a bit picky about column names. It requires:
- `ds`: A Datetime column.
- `y`: The numeric value we want to predict.

In [4]:
# Rename columns
df_prophet = df.rename(columns={'Month': 'ds', 'Passengers': 'y'})

# Convert 'ds' to datetime objects
df_prophet['ds'] = pd.to_datetime(df_prophet['ds'])

df_prophet.head()

,ds,y
0,1949-01-01,112
1,1949-02-01,118
2,1949-03-01,132
3,1949-04-01,129
4,1949-05-01,121


## 4. Building the Model
We initialize Prophet and 'fit' it to our data. Prophet automatically detects the best parameters for trend and seasonality.

In [5]:
m = Prophet(interval_width=0.95) # 95% confidence interval
m.fit(df_prophet)

15:24:04 - cmdstanpy - INFO - Chain [1] start processing
15:24:05 - cmdstanpy - INFO - Chain [1] done processing


## 5. Forecasting the Future
We will now ask the model to predict the next **24 months**.

In [6]:
# Create a dataframe with future dates
future = m.make_future_dataframe(periods=24, freq='MS') # MS = Month Start

# Predict
forecast = m.predict(future)

# Look at the forecast results
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

,ds,yhat,yhat_lower,yhat_upper
163,1962-08-01,614.176244,568.781858,657.959378
164,1962-09-01,566.266877,520.467265,610.940885
165,1962-10-01,530.555558,484.072771,575.343425
166,1962-11-01,497.702540,452.070418,542.418635
167,1962-12-01,527.228366,482.240010,572.129385


## 6. Visualizing the Forecast
The light blue area represents the uncertainty (the range where the value is likely to fall).

In [7]:
fig = go.Figure()

# Actual Data
fig.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['y'], name='Actual', mode='lines', line=dict(color='white')))

# Forecast Data
fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], name='Forecast', mode='lines', line=dict(color='#00D4FF')))

# Uncertainty Interval
fig.add_trace(go.Scatter(
    x=forecast['ds'].tolist() + forecast['ds'].tolist()[::-1],
    y=forecast['yhat_upper'].tolist() + forecast['yhat_lower'].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(0, 212, 255, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=False
))

fig.update_layout(title='Passenger Forecast: Next 2 Years', template='plotly_dark')
fig.show()

## 7. Interpreting Components
This is the most important part for engineers. We can see exactly what makes up the prediction:
1. **Trend**: The underlying growth.
2. **Yearly Seasonality**: Which months are busiest (Holidays/Vacation).

In [8]:
# Plot the components (Trend and Seasonality)
from prophet.plot import plot_components_plotly
plot_components_plotly(m, forecast)